# 01 — Ingesta RAW de NYC Taxi Data a Snowflake

Este notebook descarga archivos Parquet (yellow + green) desde la fuente pública y los carga en `NYC_TAXI_P3.RAW.TRIPS_RAW`.
También carga la tabla de zonas (`TAXI_ZONE_LOOKUP`) y registra auditoría en `INGESTION_LOG`.

In [1]:
import os
import uuid
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("=" * 60)
print("CELDA 1: Inicialización Spark + Configuración Snowflake")
print("=" * 60)

# JARs de Snowflake ya están en /usr/local/spark/jars/ (instalados por Dockerfile)
spark = (SparkSession.builder
    .appName("NYC_Taxi_Ingestion")
    .getOrCreate())
print("✓ SparkSession creada")

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}

PARQUET_BASE = os.environ["PARQUET_BASE_URL"]
YEAR_START   = int(os.environ["YEAR_START"])
YEAR_END     = int(os.environ["YEAR_END"])
MONTH_START  = int(os.environ["MONTH_START"])
MONTH_END    = int(os.environ["MONTH_END"])
SERVICES     = os.environ["SERVICES"].split(",")
RUN_ID       = os.environ.get("RUN_ID", str(uuid.uuid4())[:8])

print(f"✓ Snowflake URL: {SF_OPTIONS['sfURL']}")
print(f"✓ Database: {os.environ['SF_DATABASE']}.{os.environ['SF_SCHEMA_RAW']}")
print(f"✓ User: {os.environ['SF_USER']}")
print(f"✓ RUN_ID: {RUN_ID}")
print(f"✓ Range: {YEAR_START}-{MONTH_START:02d} to {YEAR_END}-{MONTH_END:02d}")
print(f"✓ Services: {SERVICES}")
print("=" * 60)

CELDA 1: Inicialización Spark + Configuración Snowflake
✓ SparkSession creada
✓ Snowflake URL: AASKQCL-YY64872.snowflakecomputing.com
✓ Database: NYC_TAXI_P3.RAW
✓ User: LOSMASPROSDEDATAMINING
✓ RUN_ID: backfill-2025-04
✓ Range: 2015-01 to 2025-12
✓ Services: ['yellow', 'green']


## Taxi Zone Lookup ingestion

In [2]:
import pandas as pd

print("=" * 60)
print("CELDA 2: Carga de Taxi Zone Lookup")
print("=" * 60)

zone_url = os.environ["TAXI_ZONE_URL"]
print(f">>> Descargando zonas desde: {zone_url}")
pdf_zones = pd.read_csv(zone_url)
df_zones = spark.createDataFrame(pdf_zones)
print(f"✓ Zonas descargadas: {df_zones.count()} filas")

print(">>> Escribiendo TAXI_ZONE_LOOKUP en Snowflake...")
(df_zones.write
    .format("snowflake")
    .options(**SF_OPTIONS)
    .option("dbtable", "TAXI_ZONE_LOOKUP")
    .mode("overwrite")
    .save())

print(f"✓ TAXI_ZONE_LOOKUP cargada exitosamente")
print("=" * 60)

CELDA 2: Carga de Taxi Zone Lookup
>>> Descargando zonas desde: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
✓ Zonas descargadas: 265 filas
>>> Escribiendo TAXI_ZONE_LOOKUP en Snowflake...
✓ TAXI_ZONE_LOOKUP cargada exitosamente


## Clustering Key para optimizar DELETEs idempotentes

In [3]:
print("=" * 60)
print("CELDA 2b: Clustering Key en TRIPS_RAW")
print("=" * 60)

cluster_sql = """
    ALTER TABLE NYC_TAXI_P3.RAW.TRIPS_RAW
    CLUSTER BY (service_type, source_year, source_month)
"""
try:
    spark._jvm.net.snowflake.spark.snowflake.Utils.runQuery(SF_OPTIONS, cluster_sql)
    print("\u2713 Clustering key aplicado: (service_type, source_year, source_month)")
except Exception as e:
    print(f"[WARN] Clustering key: {e}")
print("=" * 60)


CELDA 2b: Clustering Key en TRIPS_RAW
✓ Clustering key aplicado: (service_type, source_year, source_month)


## Columnas y funcion generate_months

In [4]:
print("=" * 60)
print("CELDA 3: Definicion de columnas y funcion generate_months")
print("=" * 60)

def generate_months(y_start, y_end, m_start, m_end):
    months = []
    for year in range(y_start, y_end + 1):
        m_lo = m_start if year == y_start else 1
        m_hi = m_end   if year == y_end   else 12
        for month in range(m_lo, m_hi + 1):
            months.append((year, month))
    return months

YELLOW_COLS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge", "Airport_fee"
]

GREEN_COLS = [
    "VendorID", "lpep_pickup_datetime", "lpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "trip_type", "ehail_fee"
]

ALL_COLS = [
    "VendorID",
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "lpep_pickup_datetime", "lpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "trip_type", "ehail_fee",
    "run_id", "service_type", "source_year", "source_month",
    "ingested_at_utc", "source_path"
]

_test_months = generate_months(YEAR_START, YEAR_END, MONTH_START, MONTH_END)
print(f"generate_months: {len(_test_months)} meses a procesar")
print(f"YELLOW_COLS: {len(YELLOW_COLS)} columnas")
print(f"GREEN_COLS: {len(GREEN_COLS)} columnas")
print(f"ALL_COLS (unificada): {len(ALL_COLS)} columnas")
print("=" * 60)


CELDA 3: Definicion de columnas y funcion generate_months
generate_months: 132 meses a procesar
YELLOW_COLS: 19 columnas
GREEN_COLS: 21 columnas
ALL_COLS (unificada): 29 columnas


## Ingest one month (idempotent via DELETE + INSERT)

In [5]:
import requests
import time
from pyspark.sql.types import TimestampNTZType, TimestampType

def ingest_one_month(service, year, month, run_id):
    source_month_str = f"{year}-{month:02d}"
    url = f"{PARQUET_BASE}/{service}_tripdata_{source_month_str}.parquet"
    local_path = f"/home/jovyan/data/{service}_{source_month_str}.parquet"
    started = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    
    # Download — skip HEAD, go straight to GET with exponential backoff
    max_retries = 5
    base_wait = 2
    success = False
    
    print(f"[START] {service} {source_month_str} \u2014 downloading...")
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, stream=True, timeout=300)
            resp.raise_for_status()
            with open(local_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    f.write(chunk)
            print(f"[DOWNLOADED] {service} {source_month_str}")
            success = True
            break
        except Exception as e:
            wait_time = base_wait * (2 ** attempt)
            print(f"[WARN] {service} {source_month_str} attempt {attempt + 1}/{max_retries} failed: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)
            
    if not success:
        print(f"[ERROR] {service} {source_month_str}: Max retries reached.")
        return {"service": service, "year": year, "month": month, "status": "error", "rows": 0}

    df = spark.read.parquet(local_path)

    # Fix: cast TimestampNTZType -> TimestampType
    for field in df.schema.fields:
        if isinstance(field.dataType, TimestampNTZType):
            df = df.withColumn(field.name, F.col(field.name).cast(TimestampType()))

    # Fix: add missing columns with proper types
    from pyspark.sql.types import DoubleType, StringType
    ts_cols = {"tpep_pickup_datetime","tpep_dropoff_datetime","lpep_pickup_datetime","lpep_dropoff_datetime"}
    numeric_cols = {"trip_type","ehail_fee"}
    for col in ALL_COLS:
        if col not in df.columns:
            if col in ts_cols:
                df = df.withColumn(col, F.lit(None).cast(TimestampType()))
            elif col in numeric_cols:
                df = df.withColumn(col, F.lit(None).cast(DoubleType()))
            else:
                df = df.withColumn(col, F.lit(None).cast(StringType()))

    now_ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    df = (df
        .withColumn("run_id",          F.lit(run_id))
        .withColumn("service_type",    F.lit(service))
        .withColumn("source_year",     F.lit(year))
        .withColumn("source_month",    F.lit(month))
        .withColumn("ingested_at_utc", F.to_timestamp(F.lit(now_ts)))
        .withColumn("source_path",     F.lit(url))
        .select(ALL_COLS)
    )

    # IDEMPOTENCY: delete existing data for this month/service
    delete_sql = f"""
        DELETE FROM NYC_TAXI_P3.RAW.TRIPS_RAW
        WHERE service_type = '{service}'
          AND source_year  = {year}
          AND source_month = {month}
    """
    spark._jvm.net.snowflake.spark.snowflake.Utils.runQuery(SF_OPTIONS, delete_sql)

    (df.write
        .format("snowflake")
        .options(**SF_OPTIONS)
        .option("dbtable", "TRIPS_RAW")
        .mode("append")
        .save())

    # Get row count from Snowflake (avoids double-scan of parquet)
    count_sql = f"""
        SELECT COUNT(*) AS cnt FROM NYC_TAXI_P3.RAW.TRIPS_RAW
        WHERE service_type = '{service}'
          AND source_year  = {year}
          AND source_month = {month}
    """
    count_df = (spark.read.format("snowflake").options(**SF_OPTIONS)
               .option("query", count_sql).load())
    total_rows = count_df.collect()[0][0]

    finished = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    log_df = spark.createDataFrame([{
        "run_id": run_id, "service_type": service,
        "source_year": year, "source_month": month,
        "source_path": url, "rows_loaded": total_rows,
        "rows_in_source": total_rows, "status": "loaded",
        "error_message": "", "started_at_utc": started,
        "finished_at_utc": finished
    }])
    log_df = log_df.withColumn("started_at_utc", F.to_timestamp("started_at_utc")).withColumn("finished_at_utc", F.to_timestamp("finished_at_utc"))
    (log_df.write
        .format("snowflake")
        .options(**SF_OPTIONS)
        .option("dbtable", "INGESTION_LOG")
        .option("column_mapping", "name")
        .mode("append")
        .save())

    import os as _os
    _os.remove(local_path)
    print(f"[DONE] {service} {source_month_str}: {total_rows} rows loaded")
    return {"service": service, "year": year, "month": month, "status": "loaded", "rows": total_rows}


## Run the full backfill (parallelized)

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed

months = generate_months(YEAR_START, YEAR_END, MONTH_START, MONTH_END)
tasks = [(svc, y, m) for svc in SERVICES for y, m in months]
results = []
MAX_WORKERS = 4

print(f"=== BACKFILL: {len(tasks)} tasks with {MAX_WORKERS} workers ===")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(ingest_one_month, svc, y, m, RUN_ID): (svc, y, m)
              for svc, y, m in tasks}
    for future in as_completed(futures):
        key = futures[future]
        try:
            result = future.result()
            results.append(result)
        except Exception as e:
            svc, y, m = key
            print(f"[EXCEPTION] {svc} {y}-{m:02d}: {e}")
            results.append({"service": svc, "year": y, "month": m, "status": "error", "rows": 0})

import pandas as pd
df_results = pd.DataFrame(results)
print(f"\n=== INGESTION SUMMARY ===")
print(f"Total months attempted: {len(df_results)}")
loaded = len(df_results[df_results['status']=='loaded'])
skipped = len(df_results[df_results['status']=='skipped'])
errors = len(df_results[df_results['status']=='error'])
print(f"Loaded:  {loaded}")
print(f"Skipped: {skipped}")
print(f"Errors:  {errors}")
total_rows_all = df_results["rows"].sum()
print(f"Total rows ingested: {total_rows_all:,.0f}")
df_results


=== BACKFILL: 264 tasks with 4 workers ===
[START] yellow 2015-01 — downloading...
[START] yellow 2015-02 — downloading...
[START] yellow 2015-03 — downloading...
[START] yellow 2015-04 — downloading...
[DOWNLOADED] yellow 2015-02
[DOWNLOADED] yellow 2015-01
[DOWNLOADED] yellow 2015-03
[DOWNLOADED] yellow 2015-04
[DONE] yellow 2015-01: 12741035 rows loaded
[START] yellow 2015-05 — downloading...
[DONE] yellow 2015-04: 13063758 rows loaded
[START] yellow 2015-06 — downloading...
[DONE] yellow 2015-03: 13342951 rows loaded
[START] yellow 2015-07 — downloading...
[DOWNLOADED] yellow 2015-05
[DOWNLOADED] yellow 2015-07
[DOWNLOADED] yellow 2015-06
[DONE] yellow 2015-02: 12442394 rows loaded
[START] yellow 2015-08 — downloading...
[DOWNLOADED] yellow 2015-08
[DONE] yellow 2015-07: 11559666 rows loaded
[START] yellow 2015-09 — downloading...
[DOWNLOADED] yellow 2015-09
[DONE] yellow 2015-06: 12324936 rows loaded
[START] yellow 2015-10 — downloading...
[DONE] yellow 2015-05: 13157677 rows load

,service,year,month,status,rows
0,yellow,2015,1,loaded,12741035
1,yellow,2015,4,loaded,13063758
2,yellow,2015,3,loaded,13342951
3,yellow,2015,2,loaded,12442394
4,yellow,2015,7,loaded,11559666
...,...,...,...,...,...
259,green,2025,8,error,0
260,green,2025,9,error,0
261,green,2025,10,error,0
262,green,2025,11,error,0


## Probar un mes individual
Descomenta y ejecuta la siguiente celda para probar la ingesta de un solo mes de forma manual.

In [10]:
# TEST INDIVIDUAL MONTH
result = ingest_one_month("green", 2022, 11, RUN_ID)
print(result)

[START] green 2022-11 — downloading...
[DOWNLOADED] green 2022-11
[DONE] green 2022-11: 62313 rows loaded
{'service': 'green', 'year': 2022, 'month': 11, 'status': 'loaded', 'rows': Decimal('62313')}
